In [1]:
# common imports

import sys
sys.path.append("../../ReqSeek/")

import os
import time
import math
import random
import mapper
import datasets
import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', None)

import tensorflow as tf
from pathlib import Path
from tensorflow import keras
from statistics import NormalDist

seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)
random.seed(seed)

# Loading ReqSeek

In [2]:
def ReqSeekPredict(batch):
    
    # Timing ReqSeek prediction time per batch
    start = time.perf_counter()

    encoded = tokenizer(batch["text"], padding = "max_length", max_length = 256, truncation = True, return_tensors = "tf")
    output = ReqSeek(input_ids = encoded["input_ids"], attention_mask = encoded["attention_mask"], training = False)
    predicted_labels = tf.argmax(output.logits, axis = -1).numpy()
    
    end = time.perf_counter()
    
    batch_time = end - start
    per_item_time = batch_time / len(batch["text"])
    
    return {"ReqSeek_ann": mapper.map([ReqSeek.config.id2label[int(lbl)] for lbl in predicted_labels]), 
            "ReqSeek_time": [per_item_time] * len(batch["text"])}

In [3]:
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

ReqSeek_path = '../../ReqSeek/'

tokenizer = AutoTokenizer.from_pretrained(ReqSeek_path)
ReqSeek = TFAutoModelForSequenceClassification.from_pretrained(ReqSeek_path)

Metal device set to: Apple M4 Pro


2026-06-15 20:07:06.808458: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-06-15 20:07:06.808576: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
All model checkpoint layers were used when initializing TFRobertaForSequenceClassification.

All the layers of TFRobertaForSequenceClassification were initialized from the model checkpoint at ../../ReqSeek/.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaForSequenceClassification for predictions without further training.


# Sampling Data for Annotation

In [4]:
# Loading the dataset  
ARID = datasets.load_from_disk('../../datasets_with_data/ARID_supporting_scripts/5_1_training_set')
ARID

DatasetDict({
    train: Dataset({
        features: ['REQID', 'REQID_expanded', 'Requirement Sentences', 'Open/ Closed Source', 'class', 'signal_keyword', 'Source', 'label'],
        num_rows: 1916
    })
    test: Dataset({
        features: ['REQID', 'REQID_expanded', 'Requirement Sentences', 'Open/ Closed Source', 'class', 'signal_keyword', 'Source', 'label'],
        num_rows: 480
    })
})

In [5]:
# Isolating test set to draw sample
first_author_annotations = mapper.map(ARID['test']['signal_keyword'])
for_human_annotation = ARID['test'].add_column('1stauth_ann', first_author_annotations)
for_human_annotation = for_human_annotation.to_pandas()[['REQID', 'REQID_expanded', 'Requirement Sentences', '1stauth_ann']]

In [6]:
# Sampling data for annotation
seed = 50
rounded_up_sample_size = 102
class_col = "1stauth_ann"

n_classes = for_human_annotation[class_col].nunique()
per_class = rounded_up_sample_size // n_classes

sample_for_human_annotation = (
    for_human_annotation.sort_index()
        .groupby(class_col, group_keys = False)
        .sample(n = per_class, random_state = seed)
        .sample(frac = 1, random_state = seed)
        .reset_index(drop = True)
)

print(sample_for_human_annotation[class_col].value_counts())
print(len(sample_for_human_annotation))

1stauth_ann
requirement                 34
contextual_auxiliary        34
system_related_auxiliary    34
Name: count, dtype: int64
102


In [7]:
# Renaming column for Label Studio; column names with white space error issue
sample_for_human_annotation = sample_for_human_annotation.rename(columns = {"Requirement Sentences": "text"})

In [8]:
# Saving sampled data to be loaded on Label Studio for annotaion
select_columns = ['REQID','REQID_expanded', 'text'] # excluding data labels to disterbute it to the annotators
sample_for_human_annotation[select_columns].to_csv("./human_annotated_data/for_human_annotation_sample_102.csv", index = False, encoding = "utf-8")

# ReqSeek Annotation

In [9]:
sample_for_ReqSeek_annotation = datasets.Dataset.from_pandas(sample_for_human_annotation[select_columns])
sample_for_ReqSeek_annotation = sample_for_ReqSeek_annotation.map(ReqSeekPredict, batched = True, batch_size = 8)       

INFO:tensorflow:Assets written to: ram://d9f36fca-3afb-4c88-9dba-9870936e4bd1/assets


INFO:tensorflow:Assets written to: ram://d9f36fca-3afb-4c88-9dba-9870936e4bd1/assets


Map:   0%|          | 0/102 [00:00<?, ? examples/s]

In [10]:
sample_for_ReqSeek_annotation.to_pandas().head(5)

,REQID,REQID_expanded,text,ReqSeek_ann,ReqSeek_time
0,2158,1912,The system must accommodate orders for compounded drugs or drug mixtures that may contain more than one NDC number for the ingredients.,requirement,0.02421
1,41250,29660,Fossi (Forecast System for Stock Pool Optimisation and Information) functionalities (equivalent to current MBKS DCVD version) will be integrated in SSS.,requirement,0.02421
2,30005-wikipedia,30005,"Also preserved fish, preserved poultry, offal, avocado, banana skins, broad bean pods and pickled or smoked herring must be avoided.",contextual_auxiliary,0.02421
3,2541,2303,The system should provide a way to merge allergies that have been entered in a duplicate patient chart.,requirement,0.02421
4,24827,13332,The Operational radio shall conform to IP 54 [IEC 529/EN 60529] as a minimum.,requirement,0.02421


In [11]:
reqseek_annotations = sample_for_ReqSeek_annotation.to_pandas()
reqseek_annotations = reqseek_annotations[['ReqSeek_ann', 'ReqSeek_time']]

# Loading Annotation Data

In [12]:
# Helper function to load annotators data
def annotated_data_loader(base_dir):
    
    base_dir = Path(base_dir)
    csv_dfs, excel_dfs = {}, {}
    excel_extensions = {".xlsx", ".xls"}

    for directory in base_dir.iterdir():
        if not directory.is_dir():
            continue

        csv_files = [
            file for file in directory.iterdir()
            if file.is_file()
            and file.suffix.lower() == ".csv"
            and not file.name.startswith(".")
        ]

        excel_files = [
            file for file in directory.iterdir()
            if file.is_file()
            and file.suffix.lower() in excel_extensions
            and not file.name.startswith(".")
            and not file.name.startswith("~$")
        ]

        if not csv_files:
            print(f"No CSV found in {directory.name}")
            continue

        if not excel_files:
            print(f"No Excel file found in {directory.name}")
            continue

        csv_file = csv_files[0]
        excel_file = excel_files[0]

        print(f"Loading directory: {directory.name}")
        print(f"  CSV:   {csv_file.name}")
        print(f"  Excel: {excel_file.name}")

        csv_dfs[directory.name] = pd.read_csv(csv_file)

        if excel_file.suffix.lower() == ".xlsx":
            excel_dfs[directory.name] = pd.read_excel(excel_file, engine = "openpyxl")
        elif excel_file.suffix.lower() == ".xls":
            excel_dfs[directory.name] = pd.read_excel(excel_file, engine = "xlrd")

    return csv_dfs, excel_dfs

In [13]:
import warnings

# Ignore excel warnings
warnings.filterwarnings("ignore", message = "Data Validation extension is not supported and will be removed")

annotator_data = annotated_data_loader('./human_annotated_data')[0]

Loading directory: A3
  CSV:   A3.csv
  Excel: A3.xlsx
Loading directory: A2
  CSV:   A2.csv
  Excel: A2.xlsx
Loading directory: A1
  CSV:   A1.csv
  Excel: A1.xlsx
No Excel file found in .ipynb_checkpoints


In [14]:
# Function to extract annotations and time to per annotator
import pandas as pd


def extract_annotation_and_lead_time(df_dict, annotation_col = "sentiment", lead_time_col = "lead_time"):
    extracted = []

    for key, df in df_dict.items():
        required_columns = [annotation_col, lead_time_col]
        missing_columns = [col for col in required_columns if col not in df.columns]

        if missing_columns:
            print(f"Skipping {key}: missing columns {missing_columns}")
            continue

        temp = df[[annotation_col, lead_time_col]].copy()
        temp = temp.rename(columns = { annotation_col: f"{key}_ann", lead_time_col: f"{key}_time"})
        extracted.append(temp.reset_index(drop = True))

    if not extracted:
        return pd.DataFrame()

    return pd.concat(extracted, axis = 1)

In [15]:
annotators_data = extract_annotation_and_lead_time(annotator_data)

# Analysing All Labeled Data for RQ1

In [16]:
all_annotations_merged = pd.concat([sample_for_human_annotation, reqseek_annotations, annotators_data], axis = 1)
all_annotations_merged.head()

,REQID,REQID_expanded,text,1stauth_ann,ReqSeek_ann,ReqSeek_time,A3_ann,A3_time,A2_ann,A2_time,A1_ann,A1_time
0,2158,1912,The system must accommodate orders for compounded drugs or drug mixtures that may contain more than one NDC number for the ingredients.,requirement,requirement,0.02421,requirement,17.665,requirement,12.240,requirement,16.608
1,41250,29660,Fossi (Forecast System for Stock Pool Optimisation and Information) functionalities (equivalent to current MBKS DCVD version) will be integrated in SSS.,requirement,requirement,0.02421,requirement,14570.971,system_related_auxiliary,55.961,requirement,11.607
2,30005-wikipedia,30005,"Also preserved fish, preserved poultry, offal, avocado, banana skins, broad bean pods and pickled or smoked herring must be avoided.",contextual_auxiliary,contextual_auxiliary,0.02421,contextual_auxiliary,24.251,contextual_auxiliary,16.382,contextual_auxiliary,29.253
3,2541,2303,The system should provide a way to merge allergies that have been entered in a duplicate patient chart.,requirement,requirement,0.02421,requirement,99.147,requirement,17.655,requirement,21.049
4,24827,13332,The Operational radio shall conform to IP 54 [IEC 529/EN 60529] as a minimum.,requirement,requirement,0.02421,requirement,17.443,requirement,12.421,requirement,11.272


In [17]:
all_annotations_merged.columns.to_list()

['REQID',
 'REQID_expanded',
 'text',
 '1stauth_ann',
 'ReqSeek_ann',
 'ReqSeek_time',
 'A3_ann',
 'A3_time',
 'A2_ann',
 'A2_time',
 'A1_ann',
 'A1_time']

In [18]:
ann_cols = ['A1_ann', 'A2_ann', 'A3_ann']
all_annotations_merged[all_annotations_merged[ann_cols].nunique(axis = 1) == 3] # Identify instances with complete disagreement among the three annotators.

,REQID,REQID_expanded,text,1stauth_ann,ReqSeek_ann,ReqSeek_time,A3_ann,A3_time,A2_ann,A2_time,A1_ann,A1_time
35,2736,2509,"Capability to enable access to health information in extra-ordinary situations during which an individual may need emergency care but due to their health status is incapable of granting access permissions, or in the case of a public health emergency during which the health status of a population needs to be determined.",system_related_auxiliary,system_related_auxiliary,0.012494,contextual_auxiliary,127.44,requirement,62.311,system_related_auxiliary,55.181


In [19]:
# Removing instances with complete disagreement among the three annotators.
mask_no_majority = all_annotations_merged[ann_cols].nunique(axis = 1) == 3
all_annotations_merged = all_annotations_merged.loc[~mask_no_majority]

In [20]:
# Two outlier labeling times were identified for annotator A3. 
# For each affected instance, the outlier time was replaced with the average labeling time of the two other annotators for the same instance.

threshold = 300  # 5 minutes in seconds

def replace_long_times_with_other_average(row):
    for col in time_cols:
        if pd.notna(row[col]) and row[col] > threshold:
            original_seconds = row[col]
            original_minutes = original_seconds / 60
            original_hours = original_seconds / 3600
            other_cols = [c for c in time_cols if c != col]
            valid_other_times = row[other_cols]
            valid_other_times = valid_other_times[valid_other_times.notna() & (valid_other_times <= threshold)]

            if valid_other_times.empty:
                print(f"\n--- row index: {row.name} ---")
                print(f"{col} was longer than 5 minutes")
                print(
                    f"Original: {original_seconds:.2f} seconds"
                    f"= {original_minutes:.2f} minutes"
                    f"= {original_hours:.2f} hours"
                )
                print("No valid replacement available, keeping original value.")

                continue

            replacement_seconds = valid_other_times.mean()
            replacement_minutes = replacement_seconds / 60
            replacement_hours = replacement_seconds / 3600
            print(f"\n--- row index: {row.name} ---")
            print(f"{col} was longer than 5 minutes")
            print(
                f"Original: {original_seconds:.2f} seconds"
                f"= {original_minutes:.2f} minutes"
                f"= {original_hours:.2f} hours"
            )
            print(
                f"Replacement average: {replacement_seconds:.2f} seconds "
                f"= {replacement_minutes:.2f} minutes "
                f"= {replacement_hours:.2f} hours"
            )
            print("Other annotator times used:")
            print(valid_other_times)
            row[col] = replacement_seconds
    return row

In [21]:
time_cols = ['A1_time', 'A2_time', 'A3_time']

all_annotations_merged[time_cols] = all_annotations_merged[time_cols].apply(
    pd.to_numeric,
    errors = 'coerce'
)

In [22]:
all_annotations_merged  = all_annotations_merged.apply(
    replace_long_times_with_other_average,
    axis = 1
)


--- row index: 1 ---
A3_time was longer than 5 minutes
Original: 14570.97 seconds= 242.85 minutes= 4.05 hours
Replacement average: 33.78 seconds = 0.56 minutes = 0.01 hours
Other annotator times used:
A1_time    11.607
A2_time    55.961
Name: 1, dtype: object

--- row index: 34 ---
A3_time was longer than 5 minutes
Original: 839.53 seconds= 13.99 minutes= 0.23 hours
Replacement average: 31.82 seconds = 0.53 minutes = 0.01 hours
Other annotator times used:
A1_time    56.228
A2_time     7.418
Name: 34, dtype: object


In [23]:
# Create consensus label using majority vote
all_annotations_merged['consensus_ann'] = (
    all_annotations_merged[ann_cols]
    .mode(axis = 1)[0]
)

# Create consensus time as the average annotation time across the three annotators
all_annotations_merged['consensus_time'] = (
    all_annotations_merged[time_cols]
    .mean(axis = 1)
)

In [24]:
all_annotations_merged.columns

Index(['REQID', 'REQID_expanded', 'text', '1stauth_ann', 'ReqSeek_ann',
       'ReqSeek_time', 'A3_ann', 'A3_time', 'A2_ann', 'A2_time', 'A1_ann',
       'A1_time', 'consensus_ann', 'consensus_time'],
      dtype='object')

# Computing Inter-rater Agreement

In [25]:
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa


def compute_inter_rater_agreement(df, prediction_cols = None, human_cols = None, ground_truth_col = "consensus_ann", fleiss_row_name = "F_kappa_ann_A_123"):
    if prediction_cols is None:
        prediction_cols = ["1stauth_ann", "ReqSeek_ann", "A1_ann", "A2_ann", "A3_ann"]

    if human_cols is None:
        human_cols = ["A1_ann", "A2_ann", "A3_ann"]

    required_cols = list(set(prediction_cols + human_cols + [ground_truth_col]))
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing columns in dataframe: {missing_cols}")

    results = []

    # Cohen's kappa vs consensus
    for col in prediction_cols:
        eval_df = df[[ground_truth_col, col]].dropna().copy()

        y_true = eval_df[ground_truth_col].astype(str).str.strip()
        y_pred = eval_df[col].astype(str).str.strip()

        results.append({
            "source": col,
            "n_compared": len(eval_df),
            "cohen_kappa_vs_consensus": cohen_kappa_score(y_true, y_pred),
            "F_kappa_ann_A_123": np.nan
        })

    results_df = pd.DataFrame(results)

    # Fleiss' kappa among human annotators
    fleiss_df = df[human_cols].dropna().copy()

    for col in human_cols:
        fleiss_df[col] = fleiss_df[col].astype(str).str.strip()

    ratings_matrix, categories = aggregate_raters(fleiss_df.to_numpy())
    fleiss_value = fleiss_kappa(ratings_matrix, method = "fleiss")

    fleiss_row = {
        "source": fleiss_row_name,
        "n_compared": len(fleiss_df),
        "cohen_kappa_vs_consensus": np.nan,
        "F_kappa_ann_A_123": fleiss_value
    }

    results_df = pd.concat([results_df, pd.DataFrame([fleiss_row])], ignore_index = True)

    return results_df

In [26]:
compute_inter_rater_agreement(all_annotations_merged).round(2)

,source,n_compared,cohen_kappa_vs_consensus,F_kappa_ann_A_123
0,1stauth_ann,101,0.84,NaN
1,ReqSeek_ann,101,0.81,NaN
2,A1_ann,101,0.85,NaN
3,A2_ann,101,0.86,NaN
4,A3_ann,101,0.80,NaN
5,F_kappa_ann_A_123,101,NaN,0.67


# Computing Berry Style Evaluation

In [27]:
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import fbeta_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# This function is to compute some metrics based on the Berry's paper: https://link.springer.com/chapter/10.1007/978-3-031-73143-3_14

def multiclass_berry_evaluation(df, gold_col, pred_col, time_col = None, system_name = "annotator"):
    
    '''
        **Implementation note:** This function was written by Kasra in full ''deadline goblin'' mode. 
        As a software engineer, he is legally required to be mildly ashamed of how long this function is. 
        Yes, it could be split into smaller functions. Yes, it could be prettier. No, that did not happen. 
        The purpose here is not to win a clean-code award; it is to address a reviwers concern of the RQ1 before the time runs out. 
        Correctness was checked. Elegance is a TODO that he will never do. 
        At least he commented some parts, so if he ever returns to this code, there is a small chance he will understand what he himself wrote.
    '''
    
    df_eval = df.copy()   
    df_eval[[gold_col, pred_col]] = df_eval[[gold_col, pred_col]].apply(lambda s: s.astype(str).str.strip())

    y_true = df_eval[gold_col]
    y_pred = df_eval[pred_col]
    
    y_true_binary = y_true.apply(lambda x: "requirement" if x == "requirement" else "auxiliary")
    y_pred_binary = y_pred.apply(lambda x: "requirement" if x == "requirement" else "auxiliary")
    
    labels = sorted(set(y_true) | set(y_pred))
    
    summary = {
        "annotator": system_name,
        "gold_col": gold_col,
        "pred_col": pred_col,
        "time_col": time_col if time_col is not None and time_col in df_eval.columns else None,
        "compared_items": len(df_eval),
        "correct_count": (y_true == y_pred).sum(),
        "incorrect_count": (y_true != y_pred).sum(),
        "trinary_P": precision_score(y_true, y_pred, average = "weighted"),
        "trinary_R": recall_score(y_true, y_pred, average = "weighted"),
        "trinary_F1": f1_score(y_true, y_pred, average = "weighted"),
        "trinary_F0.89": fbeta_score(y_true, y_pred, beta = 0.89, average = "weighted"),
        
        "binary_P": precision_score(y_true_binary, y_pred_binary, pos_label= 'requirement', average = "binary"),
        "binary_R": recall_score(y_true_binary, y_pred_binary, pos_label= 'requirement', average = "binary"),
        "binary_F1": f1_score(y_true_binary, y_pred_binary, pos_label= 'requirement', average = "binary"),
        "binary_F0.89": fbeta_score(y_true_binary, y_pred_binary, pos_label= 'requirement', beta = 0.89, average = "binary"),
    }

    for label in labels:
        summary[f"gold_count_{label}"] = (y_true == label).sum()
        summary[f"pred_count_{label}"] = (y_pred == label).sum()
        summary[f"gold_fraction_{label}"] = (y_true == label).mean()
        summary[f"pred_fraction_{label}"] = (y_pred == label).mean()
            
    confusion_multiclass = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=labels),
        index=[f"gold_{x}" for x in labels],
        columns=[f"pred_{x}" for x in labels],
    )

    report_multiclass = pd.DataFrame(classification_report(y_true, y_pred, output_dict=True)).T

    # ------------------------------------------------------------------
    # Build all-class TP/FP/FN/TN masks, sklearn returns only the counts
    # ------------------------------------------------------------------

    tp_mask_all = pd.Series(False, index = df_eval.index)
    fp_mask_all = pd.Series(False, index = df_eval.index)
    fn_mask_all = pd.Series(False, index = df_eval.index)
    tn_mask_all = pd.Series(False, index = df_eval.index)

    # For multiclass, TP/FP/FN/TN are defined class-wise.
    # Here I aggregate the class-wise masks across all classes.
    for target_class in labels:
        class_true = y_true.eq(target_class)
        class_pred = y_pred.eq(target_class)

        tp_mask = class_true & class_pred
        fp_mask = (~class_true) & class_pred
        fn_mask = class_true & (~class_pred)
        tn_mask = (~class_true) & (~class_pred)

        tp_mask_all = tp_mask_all | tp_mask
        fp_mask_all = fp_mask_all | fp_mask
        fn_mask_all = fn_mask_all | fn_mask
        tn_mask_all = tn_mask_all | tn_mask

    summary.update({"TP": tp_mask_all.sum(), "FP": fp_mask_all.sum(), "FN": fn_mask_all.sum(), "TN": tn_mask_all.sum()})

    confusion_pair_summary = None

    if time_col is not None and time_col in df_eval.columns:
        df_eval[time_col] = pd.to_numeric(df_eval[time_col], errors = "coerce")

        minutes_col = f"{time_col}_minutes"
        df_eval[minutes_col] = df_eval[time_col] / 60

        summary.update({
            "total_time_seconds": df_eval[time_col].sum(),
            "total_time_minutes": df_eval[minutes_col].sum(),
            "mean_time_per_inst_seconds": df_eval[time_col].mean(),
            "median_time_per_inst_seconds": df_eval[time_col].median(),
            "std_time_per_inst_seconds": df_eval[time_col].std(),
            "min_time_per_inst_seconds": df_eval[time_col].min(),
            "max_time_per_inst_seconds": df_eval[time_col].max(),
            "mean_time_per_inst_minutes": df_eval[minutes_col].mean(),
            "median_time_per_inst_minutes": df_eval[minutes_col].median(),
            "std_time_per_inst_minutes": df_eval[minutes_col].std(),
            "min_time_per_inst_minutes": df_eval[minutes_col].min(),
            "max_time_per_inst_minutes": df_eval[minutes_col].max(),
        })

        tp_times_sec = df_eval.loc[tp_mask_all, time_col]
        fp_times_sec = df_eval.loc[fp_mask_all, time_col]
        fn_times_sec = df_eval.loc[fn_mask_all, time_col]
        tn_times_sec = df_eval.loc[tn_mask_all, time_col]

        tp_times_min = df_eval.loc[tp_mask_all, minutes_col]
        fp_times_min = df_eval.loc[fp_mask_all, minutes_col]
        fn_times_min = df_eval.loc[fn_mask_all, minutes_col]
        tn_times_min = df_eval.loc[tn_mask_all, minutes_col]

        summary.update({
            "mean_time_to_find_TP_seconds": tp_times_sec.mean(),
            "median_time_to_find_TP_seconds": tp_times_sec.median(),
            "mean_time_to_decide_FP_seconds": fp_times_sec.mean(),
            "median_time_to_decide_FP_seconds": fp_times_sec.median(),
            "mean_time_to_miss_FN_seconds": fn_times_sec.mean(),
            "median_time_to_miss_FN_seconds": fn_times_sec.median(),
            "mean_time_to_decide_TN_seconds": tn_times_sec.mean(),
            "median_time_to_decide_TN_seconds": tn_times_sec.median(),
            "mean_time_to_find_TP_minutes": tp_times_min.mean(),
            "median_time_to_find_TP_minutes": tp_times_min.median(),
            "mean_time_to_decide_FP_minutes": fp_times_min.mean(),
            "median_time_to_decide_FP_minutes": fp_times_min.median(),
            "mean_time_to_miss_FN_minutes": fn_times_min.mean(),
            "median_time_to_miss_FN_minutes": fn_times_min.median(),
            "mean_time_to_decide_TN_minutes": tn_times_min.mean(),
            "median_time_to_decide_TN_minutes": tn_times_min.median(),
        })

        pair_rows = []

        for gold_label in labels:
            for pred_label in labels:
                mask = (df_eval[gold_col].eq(gold_label) & df_eval[pred_col].eq(pred_label))
                pair_rows.append({
                    "annotator": system_name,
                    "gold_label": gold_label,
                    "pred_label": pred_label,
                    "confusion_pair": f"{gold_label} -> {pred_label}",
                    "count": mask.sum(),
                    "mean_time_seconds": df_eval.loc[mask, time_col].mean(),
                    "median_time_seconds": df_eval.loc[mask, time_col].median(),
                    "mean_time_minutes": df_eval.loc[mask, minutes_col].mean(),
                    "median_time_minutes": df_eval.loc[mask, minutes_col].median(),
                })

        confusion_pair_summary = pd.DataFrame(pair_rows)

    return summary, confusion_multiclass, report_multiclass, confusion_pair_summary

In [28]:
def run_berry_evaluation(df, gold_col = "consensus_ann", pred_cols = None):
    
    summaries = []
    confusion_matrices = {}
    classification_reports = {}
    confusion_pair_summaries = {}
    
    if pred_cols is None:
        pred_cols = ["1stauth_ann", "ReqSeek_ann", "A1_ann", "A2_ann", "A3_ann"]

    for pred_col in pred_cols:
        inferred_time_col = pred_col.replace("_ann", "_time")

        if inferred_time_col not in df.columns:
            inferred_time_col = None

        summary, confusion_df, report_df, pair_summary_df = multiclass_berry_evaluation(
            df=df,
            gold_col=gold_col,
            pred_col=pred_col,
            time_col=inferred_time_col,
            system_name=pred_col,
        )

        summaries.append(summary)
        confusion_matrices[pred_col] = confusion_df
        classification_reports[pred_col] = report_df
        confusion_pair_summaries[pred_col] = pair_summary_df

    summary_table = pd.DataFrame(summaries)
    pair_summary_table = pd.concat([df_pair for df_pair in confusion_pair_summaries.values() if df_pair is not None], ignore_index = True)

    return summary_table, confusion_matrices, classification_reports, pair_summary_table

In [29]:
summary_table, conf_mats, reports, pair_summary_table = run_berry_evaluation(
    df = all_annotations_merged,
    gold_col = "consensus_ann",
    pred_cols = ["1stauth_ann", "ReqSeek_ann", "A1_ann", "A2_ann", "A3_ann"],
)

In [30]:
summary_table.columns.to_list()

['annotator',
 'gold_col',
 'pred_col',
 'time_col',
 'compared_items',
 'correct_count',
 'incorrect_count',
 'trinary_P',
 'trinary_R',
 'trinary_F1',
 'trinary_F0.89',
 'binary_P',
 'binary_R',
 'binary_F1',
 'binary_F0.89',
 'gold_count_contextual_auxiliary',
 'pred_count_contextual_auxiliary',
 'gold_fraction_contextual_auxiliary',
 'pred_fraction_contextual_auxiliary',
 'gold_count_requirement',
 'pred_count_requirement',
 'gold_fraction_requirement',
 'pred_fraction_requirement',
 'gold_count_system_related_auxiliary',
 'pred_count_system_related_auxiliary',
 'gold_fraction_system_related_auxiliary',
 'pred_fraction_system_related_auxiliary',
 'TP',
 'FP',
 'FN',
 'TN',
 'total_time_seconds',
 'total_time_minutes',
 'mean_time_per_inst_seconds',
 'median_time_per_inst_seconds',
 'std_time_per_inst_seconds',
 'min_time_per_inst_seconds',
 'max_time_per_inst_seconds',
 'mean_time_per_inst_minutes',
 'median_time_per_inst_minutes',
 'std_time_per_inst_minutes',
 'min_time_per_inst_

In [31]:
select_cols_4_paper = ['annotator',
                      'compared_items',
                      'gold_fraction_requirement',
                      'pred_fraction_requirement',
                      'trinary_P', 
                      'trinary_R', 
                      'trinary_F1',
                      'trinary_F0.89',
                      'total_time_minutes', 
                      'mean_time_to_find_TP_seconds', 
                      'mean_time_to_decide_FP_seconds']
summary_table[select_cols_4_paper].round(2)

,annotator,compared_items,gold_fraction_requirement,pred_fraction_requirement,trinary_P,trinary_R,trinary_F1,trinary_F0.89,total_time_minutes,mean_time_to_find_TP_seconds,mean_time_to_decide_FP_seconds
0,1stauth_ann,101,0.37,0.34,0.92,0.89,0.90,0.90,NaN,NaN,NaN
1,ReqSeek_ann,101,0.37,0.32,0.90,0.87,0.88,0.88,0.02,0.01,0.01
2,A1_ann,101,0.37,0.37,0.90,0.90,0.90,0.90,31.73,17.20,33.92
3,A2_ann,101,0.37,0.35,0.91,0.91,0.91,0.91,31.08,18.27,20.36
4,A3_ann,101,0.37,0.30,0.89,0.87,0.87,0.88,49.77,30.40,23.95


In [32]:
# Computing beta for weighted F calcuation based on Berry's paper: https://link.springer.com/chapter/10.1007/978-3-031-73143-3_14
human_beta_df = summary_table[summary_table["annotator"].isin(ann_cols)].copy()
human_beta_df["empirical_beta"] = (human_beta_df["mean_time_to_find_TP_seconds"] / human_beta_df["mean_time_to_decide_FP_seconds"])
overall_empirical_beta = human_beta_df["empirical_beta"].mean(skipna = True)

print(f"beta: {np.round(overall_empirical_beta, 2)}")

beta: 0.89


# Annotators Background

In [33]:
# Loading annotators background information
annotator_background = annotated_data_loader('./human_annotated_data')[1]

Loading directory: A3
  CSV:   A3.csv
  Excel: A3.xlsx
Loading directory: A2
  CSV:   A2.csv
  Excel: A2.xlsx
Loading directory: A1
  CSV:   A1.csv
  Excel: A1.xlsx
No Excel file found in .ipynb_checkpoints


In [34]:
annotator_info = (pd.concat([df.assign(Annotator_ID = annotator_id) for annotator_id, df in annotator_background.items()], ignore_index = True))

# Move Annotator_ID to the first column
cols = ["Annotator_ID"] + [col for col in annotator_info.columns if col != "Annotator_ID"]
annotator_info = annotator_info[cols]

In [35]:
annotator_info

,Annotator_ID,Education Level,Professional Background,Software engineering experience,Requirements engineering experience,SRS reading/specification/writing familiarity,ISO 29148 / structured requirements familiarity,Prior text/requirement annotation experience
0,A3,MSc,Software Engineering/Development,3-5 years,1-3 years,Moderate,Moderate,Yes
1,A2,MSc,Software Engineering/Development,5+ years,3-5 years,High,Moderate,Yes
2,A1,MSc,Software Engineering/Development,3-5 years,1-3 years,Moderate,Moderate,Yes
